In [23]:
!nvidia-smi

Tue May 19 08:37:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             33W /   70W |     651MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
!pip install langchain langchain_openai langsmith pandas langchain_experimental matplotlib langgraph langchain_core duckduckgo-search langchain-community chromadb langchain-ollama unstructured

In [25]:
import os
from uuid import uuid4

In [26]:
unique_id = uuid4().hex[0:8]
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = f'Adaptive RAG (JudgeLlamaa) (Using Llama3.1) - {unique_id}'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = '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'
os.environ['TAVILY_API_KEY'] = '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'

In [27]:
unique_id

'2a96a39c'

In [28]:
!apt-get update
!apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 10.5 kB in 4s (2,988 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to pr

In [29]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [30]:
import subprocess

In [31]:
process = subprocess.Popen('ollama serve', shell=True)

llm

In [32]:
from langchain_ollama import ChatOllama

In [33]:
local_llm = 'llama3.1'
llm = ChatOllama(model=local_llm, temperature=0)
llm_json_mode = ChatOllama(model=local_llm, temperature=0, format = 'json')

vectorstore

In [34]:
!unzip /content/VS_1.판결문.zip

Archive:  /content/VS_1.판결문.zip
replace VS_1.판결문/2.Validation/원천데이터/VS_1.판결문/01.민사/1981~2016/2000가단31995.xml? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [35]:
from langchain_community.document_loaders import DirectoryLoader

In [36]:
loader = DirectoryLoader('/content/VS_1.판결문/2.Validation/원천데이터/VS_1.판결문/02.형사/2021', glob='**/*.xml')
형사판결_docs = loader.load()

In [37]:
형사판결_docs[0]

Document(metadata={'source': '/content/VS_1.판결문/2.Validation/원천데이터/VS_1.판결문/02.형사/2021/2021고단4264.xml'}, page_content='\n\t\t\n판결문_형사 PDF\t\t\n\t\t\n\n\n\t\t\n2021고단4264.pdf\t\t\n\t\t\n\n\n\t\t\n\n, 1 ,\n\n인  천  지  방  법  원\n\n판          결\n\n사       건 2021고단4264  강제추행\n\n피  고  인 A\n\n검       사 00(기소), 00(공판)\n\n변  호  인 변호사 00\n\n판 결 선 고 2021. 8. 27.\n\n  \n\n주       문\n\n피고인을 벌금 700만 원에 처한다. \n\n피고인이 위 벌금을 납입하지 아니하는 경우 10만 원을 1일로 환산한 기간 피고인을 \n\n노역장에 유치한다. \n\n위 벌금에 상당한 금액의 가납을 명한다. \n\n피고인에게 40시간의 성폭력 치료프로그램 이수를 명한다. \n\n  \n\n이       유\n\n범 죄 사 실\n\n  피고인은 피해자 B(여, 18세)가 근무하는 골프장을 방문한 손님으로, 2021. 3. 16. \n\n13:50경 0000에 있는  D  골프장 내 E호 F 식당에서 음식 서빙을 하고 있던 피\n\n해자의 허벅지 부위를 손바닥으로 3회 툭툭 치듯이 만지면서  탱탱하네 라고 말하였다.\n\n\n\n, 2 ,\n\n  이로써 피고인은 폭행으로 피해자를 추행하였다.\n\n증거의 요지\n\n1. 피고인의 법정진술\n\n1. B, G에 대한 각 경찰 진술조서\n\n1. 내용증명서\n\n법령의 적용\n\n1. 범죄사실에 대한 해당법조 및 형의 선택 \n\n   형법 제298조(벌금형 선택)\n\n1. 노역장유치\n\n   형법 제70조 제1항, 제69조 제2항\n\n1. 가납명령\n\n   형사소송법 제334조 제1항\n\n1. 이수명령\n\n   성폭력범죄의 처벌 등에 관한 특례법 제1

In [38]:
len(형사판결_docs)

8

Embedding

In [39]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [40]:
model_name = 'jhgan/ko-sroberta-multitask'
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': False}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs = model_kwargs,
    encode_kwargs = encode_kwargs
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
from langchain_community.vectorstores import Chroma

In [42]:
형사판결_vectorstore = Chroma.from_documents(documents=형사판결_docs, embedding=hf, collection_name='criminal')

vextor stores 판례 찾기

In [43]:
형사판결_retriever = 형사판결_vectorstore.as_retriever(search_kwargs={'k':1})

In [46]:
retrieved_docs = 형사판결_retriever.invoke(
    '절도 사건에 관한 판례를 알려줘'
)
print(retrieved_docs[0].page_content)


		
판결문_형사 PDF		
		


		
2021고단1950.pdf		
		


		

, 1 ,

울  산  지  방  법  원

판          결

사       건 2021고단1950  절도

피  고  인 A

검       사 00(기소), 00, 00(공판)

변  호  인 변호사 00(국선)

판 결 선 고 2021. 8. 27.

  

주       문

피고인을 벌금 2,000,000원에 처한다.

피고인이 위 벌금을 납입하지 아니하는 경우 100,000원을 1일로 환산한 기간 피고인을 

노역장에 유치한다. 

위 벌금에 상당한 금액의 가납을 명한다.

  

이       유

범 죄 사 실

 범죄전력 

  피고인은 2020. 6. 26. 울산지방법원에서 도로교통법위반(음주운전)죄 등으로 징역 1

년 2월, 집행유예 3년을 선고받고 같은 해 11. 13. 위 판결이 확정되었고, 2021. 4. 30. 

위 법원에서 같은 죄 등으로 징역 1년 6월을 선고받고 같은 해 7. 23. 위 판결이 확정



- 2 -

되었다. 

 범죄사실 

  피고인은 2020. 8. 15. 22:20경 0000 앞 도로에서 술에 취한 채 그곳

에 주차되어 있던 피해자 C 소유인 시가 150만 원 상당의 (차량번호 1 생략) 오토바이

를 발견하고, 위 오토바이에 꽂혀 있던 열쇠를 돌려 시동을 걸고 이를 그대로 운행하

여 가지고 가 절취하였다.

증거의 요지

1. 피고인의 법정진술

1. C의 진술서

1. 각 사진, 112신고사건처리표, 수사보고(범행장소 확인), 000 지도

1. 판시 전과: 조회회보서, 각 처분미상전과확인결과보고, 각 판결문, 대법원 나의사건

검색

법령의 적용

1. 범죄사실에 대한 해당법조 및 형의 선택

   형법 제329조, 벌금형 선택

1. 경합범처리

   형법 제37조 후단, 제39조 제1항

1. 노역장유치

   형법 제70조 제1항, 제69조 제2항

1. 가납명령

   형사소송법 제334조

Components

In [47]:
import json
from langchain_core.messages import HumanMessage, SystemMessage

In [49]:
router_instructions = """당신은 사용자 질문을 벡터스토어 또는 웹 검색으로 라우팅하는 전문가입니다.

벡터스토어에는 형사판결과 관련된 문서들이 포함되어 있습니다.

이러한 주제에 대한 질문은 벡터스토어를 사용합니다. 그 외 모든 질문, 특히 최신 사건에 대해서는 웹 검색을 사용합니다.

'websearsch' 또는 'vectorstore' 중 하나의 값을 갖는 'datasource'라는 키를 가진 JSON을 반환하세요.
"""

test_vector_sotre_1 = llm_json_mode.invoke(
    [SystemMessage(content=router_instructions)]
    + [HumanMessage(content='절도 사건에 관한 판례를 알려줘')]
)
test_vector_sotre_2 = llm_json_mode.invoke(
    [SystemMessage(content=router_instructions)]
    + [HumanMessage(content='성폭력 사건에 관한 판례를 알려줘')]
)
test_web_search = llm_json_mode.invoke(
    [SystemMessage(content=router_instructions)]
    + [HumanMessage(content='영화 군체 상영일 언제야?')]
)

print(
    json.loads(test_vector_sotre_1.content),
    json.loads(test_vector_sotre_2.content),
    json.loads(test_web_search.content),
)

ConnectError: [Errno 111] Connection refused